# 원/달러 환율(USD/KRW) 2020~2026 트렌드 분석

- 데이터: FRED `DEXKOUS` (Fed H.10, 뉴욕 정오 기준 원/달러 현물환율), `collect_data.py`로 수집
- 순서: ① 데이터 확인 → ② 정제(결측·이상치) → ③ 추세(이동평균) → ④ 변화율·급등락 → ⑤ 변동성 → ⑥ 연도별 통계 → ⑦ 월별·요일별 패턴 → ⑧ (보너스) 베이스라인 예측
- 그래프는 `images/` 폴더에 png로 저장되어 REPORT.md에서 사용됩니다.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import koreanize_matplotlib  # 한글 폰트(나눔고딕) 자동 설정

IMG = Path("images"); IMG.mkdir(exist_ok=True)
STATS = {}  # 리포트에 쓰는 수치를 모아 stats.json으로 저장

# 색: 하나의 팔레트를 역할별로 고정해서 사용
C_PRICE, C_MA20, C_MA60, C_MA120 = "#9a9893", "#eb6834", "#1baf7a", "#2a78d6"
C_UP, C_DOWN, C_NEUTRAL = "#e34948", "#2a78d6", "#9a9893"   # 환율 상승(원화 약세)=빨강, 하락=파랑
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e1"

plt.rcParams.update({
    "figure.figsize": (11, 4.8), "figure.dpi": 110, "savefig.dpi": 150,
    "axes.facecolor": "#fcfcfb", "figure.facecolor": "#fcfcfb",
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlecolor": INK, "axes.titlelocation": "left",
    "axes.unicode_minus": False, "legend.frameon": False,
})
def save(fig, name):
    fig.tight_layout(); fig.savefig(IMG / name, bbox_inches="tight"); plt.show()

ModuleNotFoundError: No module named 'distutils'

## 1. 데이터 기본 정보 확인

In [ ]:
raw = pd.read_csv("data/usdkrw_daily_raw.csv", parse_dates=["date"])
print(raw.info())
print(raw.head())
print("기간:", raw["date"].min().date(), "~", raw["date"].max().date())
print("행 수:", len(raw), "/ 결측치:", raw["usdkrw"].isna().sum())
print("날짜 중복:", raw["date"].duplicated().sum(), "/ 정렬 여부:", raw["date"].is_monotonic_increasing)

## 2. 정제: 결측치·이상치

**결측치 기준**: 이 데이터는 미국 영업일 기준이라 주말은 행 자체가 없고, 미국 공휴일(신정, 추수감사절 등)은 행은 있지만 값이 비어 있다.
거래가 없었던 날이므로 값을 채우지(ffill) 않고 **삭제**한다. 채우면 '변화율 0%'인 가짜 날이 생겨 변동성이 과소평가되기 때문이다.

In [ ]:
na_rows = raw[raw["usdkrw"].isna()]
print("결측 날짜 예시:", na_rows["date"].dt.strftime("%Y-%m-%d").head(8).tolist())
print("결측 날짜 요일 분포:", na_rows["date"].dt.day_name().value_counts().to_dict())

df = raw.dropna().set_index("date").sort_index()
STATS["n_raw"] = int(len(raw)); STATS["n_missing"] = int(raw["usdkrw"].isna().sum()); STATS["n"] = int(len(df))
STATS["start"] = str(df.index.min().date()); STATS["end"] = str(df.index.max().date())
print("정제 후 데이터 포인트:", len(df))

**이상치 기준**: 일간 변화율(전 영업일 대비 %)의 z-score가 |z| > 4 이면 후보로 표시한다.
후보가 '입력 오류'인지 '실제 급변'인지는 **다음 날 값이 원래 수준으로 되돌아오는지**로 판단한다.
오류라면 하루 튀었다가 바로 복귀(스파이크)하고, 실제 급변이라면 새 수준이 유지된다.

In [ ]:
df["ret"] = df["usdkrw"].pct_change() * 100          # 일간 변화율(%)
z = (df["ret"] - df["ret"].mean()) / df["ret"].std()
cand = df[z.abs() > 4].copy()
# 다음 영업일에 반대 방향으로 80% 이상 되돌렸으면 '스파이크(오류 의심)'
nxt = df["ret"].shift(-1).loc[cand.index]
cand["next_ret"] = nxt
cand["spike_suspect"] = (np.sign(cand["ret"]) != np.sign(nxt)) & (nxt.abs() >= 0.8 * cand["ret"].abs())
cand["gap_days"] = df.index.to_series().diff().dt.days.loc[cand.index]
print(cand[["usdkrw", "ret", "next_ret", "gap_days", "spike_suspect"]].round(2))
STATS["outlier_candidates"] = int(len(cand)); STATS["spike_suspects"] = int(cand["spike_suspect"].sum())

→ 후보는 모두 다음 날 되돌림이 없어 **실제 시장 움직임**으로 판단하고 **제거하지 않는다** (환율 분석에서 급변 자체가 분석 대상이므로).
또한 수치 범위(약 1,080~1,560원)도 정상 범위라 단위 오류 등은 없다.

## 3. 추세: 이동평균 (20일 ≈ 1개월, 60일 ≈ 1분기, 120일 ≈ 반년)

In [ ]:
for w in (20, 60, 120):
    df[f"ma{w}"] = df["usdkrw"].rolling(w).mean()

fig, ax = plt.subplots()
ax.plot(df.index, df["usdkrw"], color=C_PRICE, lw=1, label="일별 환율")
ax.plot(df.index, df["ma20"], color=C_MA20, lw=1.4, label="20일 이동평균")
ax.plot(df.index, df["ma120"], color=C_MA120, lw=2, label="120일 이동평균")
for d, label in [("2020-03-19", "코로나 충격"), ("2022-03-16", "미 연준 금리 인상 시작"), ("2024-12-03", "비상계엄 사태")]:
    ax.axvline(pd.Timestamp(d), color=INK2, lw=0.8, ls=":")
    ax.text(pd.Timestamp(d), df["usdkrw"].max() * 1.005, label, fontsize=9, color=INK2, ha="center", va="bottom")
imax, imin = df["usdkrw"].idxmax(), df["usdkrw"].idxmin()
ax.scatter([imax], [df.loc[imax, "usdkrw"]], color=C_UP, s=30, zorder=5)
ax.annotate(f"최고 {df.loc[imax,'usdkrw']:,.0f}원\n({imax:%Y-%m-%d})", (imax, df.loc[imax, "usdkrw"]),
            xytext=(-95, -10), textcoords="offset points", fontsize=9, color=INK)
ax.set_title("원/달러 환율과 이동평균 (2020–2026)")
ax.set_ylabel("원 / 1달러"); ax.legend(loc="lower right", ncol=3)
ax.set_ylim(df["usdkrw"].min() * 0.97, df["usdkrw"].max() * 1.04)
save(fig, "01_trend_moving_average.png")

STATS["max"] = [str(imax.date()), round(df.loc[imax, "usdkrw"], 2)]
STATS["min"] = [str(imin.date()), round(df.loc[imin, "usdkrw"], 2)]
STATS["first"] = round(df["usdkrw"].iloc[0], 2); STATS["last"] = round(df["usdkrw"].iloc[-1], 2)
print("최고:", STATS["max"], "/ 최저:", STATS["min"], "/ 시작→끝:", STATS["first"], "→", STATS["last"])

In [ ]:
# 연도별: 120일 이동평균 위에 있던 날의 비율 (높을수록 '상승 추세가 유지된' 해)
above = (df["usdkrw"] > df["ma120"]).where(df["ma120"].notna())
above_y = above.groupby(df.index.year).mean().mul(100).round(1)
print("120일선 위에 머문 비율(%):"); print(above_y)

# 추세 전환: 20일선이 120일선을 위/아래로 교차한 시점
sig = np.sign(df["ma20"] - df["ma120"]).dropna()
cross = sig[sig.diff().abs() == 2]
print("\n20일선 × 120일선 교차 횟수:", len(cross))
print(pd.DataFrame({"방향": cross.map({1: "상향 돌파(상승 전환)", -1: "하향 돌파(하락 전환)"}),
                    "환율": df.loc[cross.index, "usdkrw"].round(1)}))
STATS["above_ma120_by_year"] = above_y.dropna().to_dict(); STATS["n_cross"] = int(len(cross))
STATS["crosses"] = [[str(i.date()), int(v)] for i, v in cross.items()]

## 4. 변화율: 일간 변화율 분포와 급등·급락일

In [ ]:
r = df["ret"].dropna()
STATS["ret_mean"] = round(r.mean(), 4); STATS["ret_std"] = round(r.std(), 3)
STATS["share_within_05"] = round((r.abs() <= 0.5).mean() * 100, 1)
STATS["n_over_15"] = int((r.abs() >= 1.5).sum())
print(r.describe().round(3))
print(f"|변화율| ≤ 0.5% 인 날: {STATS['share_within_05']}%,  |변화율| ≥ 1.5% 인 날: {STATS['n_over_15']}일")

top = r.reindex(r.abs().sort_values(ascending=False).index).head(10)
gap = df.index.to_series().diff().dt.days
top_tbl = pd.DataFrame({"변화율(%)": top.round(2), "환율": df.loc[top.index, "usdkrw"].round(1),
                        "직전 영업일과 간격(일)": gap.loc[top.index].astype(int)})
print("\n급등락 상위 10일:"); print(top_tbl)
STATS["top_moves"] = [[str(i.date()), round(v, 2), int(gap.loc[i])] for i, v in top.items()]

# 상승/하락 비대칭: 오르는 날이 더 많은데, 큰 움직임은 하락 쪽에 몰려 있는가?
big = r[r.abs() >= 1.5]
asym = {"up_days": int((r > 0).sum()), "down_days": int((r < 0).sum()),
        "big_up": int((big > 0).sum()), "big_down": int((big < 0).sum()), "skew": round(r.skew(), 3),
        "top10_down": int((top < 0).sum())}
STATS["asym"] = asym
print("\n상승/하락 비대칭:", asym)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1, 1.6]})
bins = np.arange(-4, 4.01, 0.2)
a1.hist(r[r >= 0], bins=bins, color=C_UP, alpha=.85, label="상승일", edgecolor="#fcfcfb", lw=1)
a1.hist(r[r < 0], bins=bins, color=C_DOWN, alpha=.85, label="하락일", edgecolor="#fcfcfb", lw=1)
a1.set_title("일간 변화율 분포"); a1.set_xlabel("전 영업일 대비 변화율 (%)"); a1.set_ylabel("일수"); a1.legend()
a2.bar(r.index, r.values, width=2, color=np.where(r >= 0, C_UP, C_DOWN))
for d, v in top.head(5).items():
    a2.annotate(f"{d:%y.%m.%d}\n{v:+.1f}%", (d, v), xytext=(0, 4 if v > 0 else -4), va="bottom" if v > 0 else "top",
                textcoords="offset points", ha="center", fontsize=8, color=INK)
a2.axhline(0, color=INK2, lw=0.6); a2.set_ylim(-5, 4.5)
a2.set_title("일간 변화율 추이 (상위 5개 급변일 표시)"); a2.set_ylabel("%")
save(fig, "02_daily_change.png")

## 5. 변동성: 20일 롤링 표준편차 (연율화)

In [ ]:
df["vol20"] = df["ret"].rolling(20).std() * np.sqrt(252)   # 연율화 변동성(%)
vol_y = df["vol20"].groupby(df.index.year).mean().round(2)
vmax = df["vol20"].idxmax()
STATS["vol_by_year"] = vol_y.to_dict(); STATS["vol_max"] = [str(vmax.date()), round(df.loc[vmax, "vol20"], 2)]
STATS["vol_median"] = round(df["vol20"].median(), 2)
print("연도별 평균 변동성(%):"); print(vol_y)
print("변동성 최고:", STATS["vol_max"], "/ 전체 중앙값:", STATS["vol_median"])

# 변동성 상위 10% 구간 = '고변동성 국면'
thr = df["vol20"].quantile(0.9)
hi = df["vol20"] > thr
episodes = (hi != hi.shift()).cumsum()[hi]
ep = df[hi].groupby(episodes).apply(lambda g: pd.Series({"시작": g.index.min().date(), "끝": g.index.max().date(),
                                                         "일수": len(g), "최고변동성": round(g["vol20"].max(), 1),
                                                         "구간 중 환율 변화(원)": round(g["usdkrw"].iloc[-1] - g["usdkrw"].iloc[0], 1)}))
ep = ep[ep["일수"] >= 5].reset_index(drop=True)
print(f"\n고변동성 국면(상위 10%, {thr:.1f}% 초과, 5영업일 이상):"); print(ep)
STATS["vol_p90"] = round(thr, 2); STATS["hi_vol_episodes"] = ep.astype(str).values.tolist()

fig, ax = plt.subplots()
ax.plot(df.index, df["vol20"], color=C_MA120, lw=1.4)
ax.axhline(thr, color=C_UP, lw=1, ls="--"); ax.text(df.index[-1], thr + 0.25, f"상위 10% 기준 ({thr:.1f}%)", color=C_UP, fontsize=9, ha="right", bbox=dict(fc="#fcfcfb", ec="none", pad=1))
ax.axhline(df["vol20"].median(), color=INK2, lw=0.8, ls=":")
ax.text(pd.Timestamp("2021-06-01"), df["vol20"].median() - 1.1, f"중앙값 ({df['vol20'].median():.1f}%)", color=INK2, fontsize=9, bbox=dict(fc="#fcfcfb", ec="none", pad=1))
ax.set_title("20일 롤링 변동성 (연율화)"); ax.set_ylabel("연율화 표준편차 (%)")
save(fig, "03_rolling_volatility.png")

## 6. 연도별 구간 통계

In [ ]:
yr = df["usdkrw"].groupby(df.index.year).agg(["count", "mean", "min", "max", "first", "last"])
yr["범위(max-min)"] = yr["max"] - yr["min"]
yr["연간 변화율(%)"] = (yr["last"] / yr["first"] - 1) * 100
print(yr.round(1))
STATS["yearly"] = yr.round(2).reset_index().rename(columns={"date": "year"}).to_dict("records")

fig, ax = plt.subplots()
years = sorted(df.index.year.unique())
data = [df.loc[str(y), "usdkrw"].values for y in years]
bp = ax.boxplot(data, tick_labels=[f"{y}{'*' if y == years[-1] else ''}" for y in years], patch_artist=True,
                widths=.55, medianprops=dict(color=INK, lw=1.6), flierprops=dict(marker="o", ms=3, mfc=C_NEUTRAL, mec="none"))
for b in bp["boxes"]: b.set(facecolor="#dbe7f7", edgecolor=C_MA120, lw=1.2)
for i, y in enumerate(years, 1):
    ax.text(i, yr.loc[y, "max"] + 8, f"평균 {yr.loc[y,'mean']:,.0f}", ha="center", fontsize=9, color=INK2)
ax.set_title("연도별 환율 분포 (* 2026년은 10월까지)"); ax.set_ylabel("원 / 1달러")
save(fig, "04_yearly_boxplot.png")

## 7. 계절성 확인: 월별·요일별 평균 변화율
환율에 '특정 달/요일에 오르는' 패턴이 있는지 확인한다. 막대만 보면 착시가 생기므로 **95% 신뢰구간**을 함께 그려,
구간이 0을 포함하면 "우연과 구분되지 않는다"고 본다.

In [ ]:
mret = df["usdkrw"].resample("ME").last().pct_change().mul(100).dropna()   # 월간 변화율
m_stat = mret.groupby(mret.index.month).agg(["mean", "std", "count"])
m_stat["ci"] = 1.96 * m_stat["std"] / np.sqrt(m_stat["count"])
m_stat["up_ratio"] = mret.groupby(mret.index.month).apply(lambda s: (s > 0).mean() * 100)
print("월별 평균 월간 변화율(%):"); print(m_stat.round(2))

wd_names = ["월", "화", "수", "목", "금"]
w_stat = r.groupby(r.index.dayofweek).agg(["mean", "std", "count"])
w_stat["ci"] = 1.96 * w_stat["std"] / np.sqrt(w_stat["count"])
print("\n요일별 평균 일간 변화율(%):"); print(w_stat.round(3))
STATS["monthly"] = m_stat.round(3).reset_index().to_dict("records")
STATS["weekday"] = w_stat.round(4).reset_index().to_dict("records")
STATS["months_ci_excl_zero"] = [int(m) for m, row in m_stat.iterrows() if abs(row["mean"]) > row["ci"]]
STATS["weekdays_ci_excl_zero"] = [wd_names[d] for d, row in w_stat.iterrows() if abs(row["mean"]) > row["ci"]]
print("\n신뢰구간이 0을 포함하지 않는 월:", STATS["months_ci_excl_zero"], "/ 요일:", STATS["weekdays_ci_excl_zero"])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [2, 1]})
a1.bar(m_stat.index, m_stat["mean"], color=np.where(m_stat["mean"] >= 0, C_UP, C_DOWN), width=.6)
a1.errorbar(m_stat.index, m_stat["mean"], yerr=m_stat["ci"], fmt="none", ecolor=INK2, capsize=3, lw=1)
a1.axhline(0, color=INK2, lw=.6); a1.set_xticks(range(1, 13), [f"{m}월" for m in range(1, 13)])
a1.set_title(f"월별 평균 월간 변화율 (n={int(m_stat['count'].min())}~{int(m_stat['count'].max())}년, 오차막대=95% CI)"); a1.set_ylabel("%")
a2.bar(range(5), w_stat["mean"], color=np.where(w_stat["mean"] >= 0, C_UP, C_DOWN), width=.6)
a2.errorbar(range(5), w_stat["mean"], yerr=w_stat["ci"], fmt="none", ecolor=INK2, capsize=3, lw=1)
a2.axhline(0, color=INK2, lw=.6); a2.set_xticks(range(5), wd_names)
a2.set_title("요일별 평균 일간 변화율"); a2.set_ylabel("%")
save(fig, "05_month_weekday_pattern.png")

## 8. (보너스 B) 베이스라인 예측
마지막 60영업일을 '미래'로 숨기고, 그 직전까지의 데이터만으로 세 가지 단순 방법을 비교한다.
목적은 정확도 경쟁이 아니라 **어떤 가정이 깨지는지** 보는 것이다.

| 방법 | 가정 |
|---|---|
| Naive(마지막 값 유지) | 내일 환율 = 오늘 환율 (랜덤워크) |
| Drift(직전 1년 추세 연장) | 최근 1년의 평균 기울기가 계속된다 |
| 20일 이동평균 유지 | 최근 한 달 평균 수준으로 돌아간다 |

In [ ]:
H = 60
train, test = df["usdkrw"].iloc[:-H], df["usdkrw"].iloc[-H:]
last = train.iloc[-1]
lookback = train.iloc[-252:]
slope = (lookback.iloc[-1] - lookback.iloc[0]) / (len(lookback) - 1)
fc = pd.DataFrame({
    "Naive": np.repeat(last, H),
    "Drift(1년 추세)": last + slope * np.arange(1, H + 1),
    "20일 평균": np.repeat(train.iloc[-20:].mean(), H),
}, index=test.index)
mae = fc.apply(lambda c: (c - test).abs().mean()).round(1)
mape = fc.apply(lambda c: ((c - test).abs() / test).mean() * 100).round(2)
res = pd.DataFrame({"MAE(원)": mae, "MAPE(%)": mape}); print(res)
print("예측 시작일:", test.index[0].date(), "/ 학습 마지막 값:", round(last, 1), "/ 실제 마지막 값:", round(test.iloc[-1], 1))
STATS["forecast"] = {"start": str(test.index[0].date()), "train_last": round(last, 2), "test_last": round(test.iloc[-1], 2),
                     "slope_per_day": round(slope, 3), "metrics": res.reset_index().to_dict("records")}

# 같은 방식으로 과거 여러 시점에서 반복(롤링 백테스트) → 한 번의 결과가 우연인지 확인
starts = range(300, len(df) - H, 20)
bt = []
for s in starts:
    tr, te = df["usdkrw"].iloc[:s], df["usdkrw"].iloc[s:s + H]
    lb = tr.iloc[-252:]; sl = (lb.iloc[-1] - lb.iloc[0]) / (len(lb) - 1)
    bt.append({"Naive": (te - tr.iloc[-1]).abs().mean(),
               "Drift(1년 추세)": (te - (tr.iloc[-1] + sl * np.arange(1, len(te) + 1))).abs().mean(),
               "20일 평균": (te - tr.iloc[-20:].mean()).abs().mean()})
bt = pd.DataFrame(bt)
wins = bt.idxmin(axis=1).value_counts()
print(f"\n롤링 백테스트 {len(bt)}회 평균 MAE(원):"); print(bt.mean().round(1)); print("가장 정확했던 횟수:", wins.to_dict())
STATS["backtest"] = {"n": int(len(bt)), "mean_mae": bt.mean().round(2).to_dict(), "wins": wins.to_dict()}

fig, ax = plt.subplots()
ctx = df["usdkrw"].iloc[-H - 180:-H]
ax.plot(ctx.index, ctx, color=C_PRICE, lw=1.2, label="학습 구간")
ax.plot(test.index, test, color=INK, lw=2, label="실제(숨긴 60일)")
for (name, col), c, ls in zip(fc.items(), [C_MA20, C_MA120, C_MA60], ["-", "--", ":"]):
    ax.plot(col.index, col, color=c, lw=1.8, ls=ls, label=f"{name} (MAE {mae[name]:.0f}원)")
ax.axvline(test.index[0], color=INK2, lw=.8, ls=":")
ax.set_title("베이스라인 예측 비교: 마지막 60영업일"); ax.set_ylabel("원 / 1달러"); ax.legend(loc="lower left", fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%y-%m"))
save(fig, "06_baseline_forecast.png")

In [ ]:
Path("stats.json").write_text(json.dumps(STATS, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
df.to_csv("data/usdkrw_daily_clean.csv")
print("저장 완료: stats.json, data/usdkrw_daily_clean.csv, images/*.png")